# Experiment 1 · Go further: the complete-basis-set limit

In Experiment 1 you saw the error change as the basis grows from double- to quadruple-ζ. But what would an **infinitely large** basis give? We can't compute that directly, but we can *extrapolate* from a systematic series of basis sets.

In this notebook you will:
1. compute the (HCl)₂ → 2 HCl dissociation energy with the Dunning cc-pV*X*Z series (X = 2, 3, 4) using **PySCF**, a free Python quantum-chemistry program that runs right here in your browser;
2. extrapolate the series to the complete-basis-set (CBS) limit;
3. do the same with **your own Q-Chem numbers**.

> **How to use:** run each cell in order with ▶ (or Shift+Enter). The first cell installs PySCF.

In [ ]:
# Install PySCF (takes ~30 s on Colab; skipped if already installed)
try:
    import pyscf
except ImportError:
    %pip install -q pyscf
    import pyscf
print("PySCF", pyscf.__version__)

# Optional: Grimme's D4 dispersion correction, needed only for methods ending in "-D4".
# The ">=1.3" pin matters: older builds of this package break PySCF.
WANT_D4 = True
if WANT_D4:
    %pip install -q "pyscf-dispersion>=1.3"


In [ ]:
import numpy as np
from pyscf import gto, scf, dft

HARTREE_TO_KCAL = 627.5095

def energy(atoms, method, basis, charge=0, mult=1, grid=(99, 590), verbose=False):
    """Single-point energy (hartree) with PySCF.
    method: 'HF', or a functional name such as 'PBE', 'B3LYP', 'wB97M-V';
            add '-D4' for Grimme's D4 dispersion (e.g. 'B3LYP-D4')."""
    mol = gto.M(atom=atoms, basis=basis, charge=charge, spin=mult - 1, verbose=0)
    m = method.upper()
    disp = None
    if m.endswith("-D4"):
        m, disp = m[:-3], "d4"
    if m == "HF":
        mf = scf.RHF(mol) if mult == 1 else scf.UHF(mol)
    else:
        mf = dft.RKS(mol) if mult == 1 else dft.UKS(mol)
        mf.xc = m.replace("-", "_") if m in ("WB97M-V", "B97M-V") else m
        mf.grids.atom_grid = grid
    if disp:
        mf.disp = disp
    mf.conv_tol = 1e-9
    e = mf.kernel()
    if not mf.converged:
        print(f"  WARNING: SCF did not converge for {method}/{basis}")
    if verbose:
        print(f"{method}/{basis}: {e:.10f} Eh, {mol.nao} basis functions")
    return e


## The molecules
Same geometries as on the Experiment 1 page (Å).

In [ ]:
DIMER = """
Cl     1.86082400    -0.06541100    -0.00006800
H      1.75394100     1.21098100     0.00034100
Cl    -1.92526600     0.00557100    -0.00009700
H     -0.65842700    -0.19370300     0.00247600
"""

HCL = """
Cl     0.00000000     0.00000000     0.00000000
H      0.00000000     0.00000000     1.27907000
"""

REF = 2.01  # kcal/mol

## 1 · Run the basis-set series
Pick a method. `PBE` is fastest (≈1 min in total on Colab), while `wB97M-V` and `B3LYP` take a few minutes. Then run the cell.

In [ ]:
METHOD = "PBE"   # try also "B3LYP", "wB97M-V" or "HF"
BASES = {2: "cc-pVDZ", 3: "cc-pVTZ", 4: "cc-pVQZ"}

dE = {}
for X, basis in BASES.items():
    e_dimer = energy(DIMER, METHOD, basis)
    e_hcl = energy(HCL, METHOD, basis)
    dE[X] = (2 * e_hcl - e_dimer) * HARTREE_TO_KCAL
    print(f"{basis:8s}  ΔE = {dE[X]:7.3f} kcal/mol   error = {dE[X] - REF:+.3f}")

## 2 · Extrapolate
For DFT and HF, energies converge roughly **exponentially** with the cardinal number *X*:

$$E(X) = E_\text{CBS} + A\,e^{-\alpha X}$$

With three points (X = 2, 3, 4) the three unknowns are determined exactly, and

$$E_\text{CBS} = \frac{E_2\,E_4 - E_3^2}{E_2 + E_4 - 2E_3}.$$

The formula only makes sense if the series converges *monotonically* (each step changes the energy less than the previous one, in the same direction). We apply it directly to the reaction energy.

In [ ]:
def cbs3(e2, e3, e4):
    d1, d2 = e3 - e2, e4 - e3
    if d1 * d2 <= 0 or abs(d2) >= abs(d1):
        print("  ⚠ the series is not monotonically converging: the extrapolation is not meaningful")
        return None
    return (e2 * e4 - e3**2) / (e2 + e4 - 2 * e3)

cbs = cbs3(dE[2], dE[3], dE[4])
if cbs is not None:
    print(f"{METHOD}/CBS  ΔE = {cbs:.3f} kcal/mol   error = {cbs - REF:+.3f}")

In [ ]:
import matplotlib.pyplot as plt
X = list(dE)
plt.plot(X, [dE[x] for x in X], "o-", label=f"{METHOD} / cc-pVXZ")
if cbs is not None:
    plt.axhline(cbs, ls="--", label=f"{METHOD} CBS estimate")
plt.axhline(REF, color="k", lw=1, label="reference")
plt.xticks(X, ["DZ", "TZ", "QZ"]); plt.ylabel("ΔE (kcal/mol)"); plt.legend(); plt.title("(HCl)₂ → 2 HCl")
plt.show()

## 3 · Now with your Q-Chem results
Paste the reaction energies you entered on the Experiment 1 page (kcal/mol) and run the cell. Try both families: does the def2 series (SVP, TZVP, QZVP) extrapolate as nicely as the cc series?

In [ ]:
# your Q-Chem ΔE values (kcal/mol); replace the None values
mine = {
    "wB97M-V cc":   {2: None, 3: None, 4: None},
    "wB97M-V def2": {2: None, 3: None, 4: None},
    "HF cc":        {2: None, 3: None, 4: None},
}
for name, s in mine.items():
    if None in s.values():
        continue
    c = cbs3(s[2], s[3], s[4])
    print(f"{name:14s} CBS ≈ {c:.3f}  error = {c - REF:+.3f}" if c is not None else f"{name:14s} (not extrapolable)")

## Questions to think about
1. How far is your quadruple-ζ result from the CBS estimate? Is the QZ basis "good enough" for this reaction?
2. After extrapolation, does the error of each method go to zero? What is left, and where does it come from?
3. HF errors in Experiment 1 did not always decrease with the basis size. Using the CBS estimate, explain why a *bigger* basis can make HF *worse* compared with the reference.
4. When would you *not* trust a three-point extrapolation?